# ALAD-Mobile — Demucs miễn phí trên Google Colab

Notebook này mở API tạm thời để ALAD-Mobile gửi từng đoạn WAV, nhận lại `vocals_wav` và `instrumental_wav`. **Phải giữ notebook đang chạy và không đóng phiên Colab.** URL Cloudflare sẽ thay đổi khi khởi động lại. Đây là giải pháp thử nghiệm miễn phí, không bảo đảm chạy liên tục hoặc độ trễ thấp.

Chỉ dùng âm thanh bạn có quyền xử lý. Endpoint công khai qua URL ngẫu nhiên; không chia sẻ URL này.


In [ ]:
# 1) Cài thư viện (chạy một lần sau mỗi phiên Colab mới)
!pip -q install demucs==4.0.1 fastapi==0.115.6 uvicorn[standard]==0.34.0 python-multipart==0.0.20 soundfile==0.13.1


In [ ]:
# 2) Tạo API tách giọng tương thích với ALAD-Mobile
from pathlib import Path
server_code = r'''
import io, base64, os
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import numpy as np
import soundfile as sf
import torch, torchaudio
from demucs.pretrained import get_model
from demucs.apply import apply_model

app = FastAPI(title="ALAD Demucs Free Colab")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["GET", "POST"], allow_headers=["*"])
MODEL = None

@app.on_event("startup")
def load_model():
    global MODEL
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 2)))
    MODEL = get_model(name="htdemucs")
    MODEL.cpu().eval()

@app.get("/health")
def health():
    return {"status": "ok", "model": "htdemucs", "device": "cpu"}

def wav_b64(audio, sr):
    buf = io.BytesIO(); sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode("ascii")

@app.post("/separate")
async def separate(file: UploadFile = File(...)):
    if MODEL is None: raise HTTPException(503, "Model is loading; retry")
    raw = await file.read()
    if not raw or len(raw) > 8 * 1024 * 1024: raise HTTPException(413, "Audio must be 1 byte to 8 MB")
    try:
        audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        x = torch.from_numpy(audio.T.copy())
        if sr != MODEL.samplerate: x = torchaudio.functional.resample(x, sr, MODEL.samplerate)
        if x.shape[0] == 1: x = x.repeat(2, 1)
        elif x.shape[0] > 2: x = x[:2]
        mean = x.mean(); std = x.std().clamp(min=1e-8); x = (x - mean) / std
        with torch.inference_mode():
            stems = apply_model(MODEL, x.unsqueeze(0), device="cpu", shifts=0, split=True, overlap=0.25, progress=False)[0]
        stems = stems * std + mean
        names = MODEL.sources
        vocals = stems[names.index("vocals")].mean(0)
        instrumental = sum(stems[i] for i,n in enumerate(names) if n != "vocals").mean(0)
        vocals = torchaudio.functional.resample(vocals.unsqueeze(0), MODEL.samplerate, 16000).squeeze(0).numpy()
        instrumental = torchaudio.functional.resample(instrumental.unsqueeze(0), MODEL.samplerate, 16000).squeeze(0).numpy()
        return {"vocals_wav": wav_b64(vocals,16000), "instrumental_wav": wav_b64(instrumental,16000), "sample_rate":16000}
    except Exception as e:
        raise HTTPException(500, f"Separation failed: {type(e).__name__}: {e}")
\n'''
Path('/content/server.py').write_text(server_code)
print('Đã tạo /content/server.py')


In [ ]:
# 3) Khởi động API và Cloudflare Tunnel miễn phí
import subprocess, time, re, requests, os, pathlib, sys

# Tải Cloudflared (không cần tài khoản Cloudflare)
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

api_log = open('/content/alad_api.log','w')
api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '7860'], cwd='/content', stdout=api_log, stderr=subprocess.STDOUT)
time.sleep(8)

tunnel = subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
start = time.time()
while time.time()-start < 120:
    line = tunnel.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0); break
    else: time.sleep(1)
if not public_url:
    raise RuntimeError('Không lấy được URL Cloudflare. Xem log phía trên và chạy lại ô này.')
print('\n=== URL DÁN VÀO ALAD-MOBILE ===')
print(public_url)
print('Health:', public_url + '/health')


## Cách dùng trong ALAD-Mobile
1. Chạy lần lượt cả 3 ô trên. Chờ ô cuối in URL `https://....trycloudflare.com`.
2. Mở URL đó cộng `/health` trong tab mới. Phải nhận JSON có `status: ok`. Lần đầu có thể chờ model tải.
3. Trong ALAD-Mobile mở **Cài đặt** → mục **Máy chủ tách giọng Demucs (Google Colab miễn phí)** → dán riêng URL gốc, không thêm `/health` hoặc `/separate`. URL được lưu tự động.
4. Bắt đầu lồng tiếng. Giữ Colab mở và máy tính không ngủ. Nếu Colab ngắt hoặc URL đổi, chạy lại ô 3 và cập nhật URL trong ứng dụng.

### Giới hạn
- Colab Free có thời lượng/nguồn lực thay đổi, có thể ngắt phiên; không phù hợp phát trực tiếp lâu dài.
- Demucs HTDemucs trên CPU có thể chậm; GPU miễn phí không phải lúc nào cũng được cấp.
- Âm thanh xử lý theo từng cửa sổ 6 giây nên sẽ có độ trễ đáng kể.
- URL tunnel là URL công khai tạm thời; không đăng công khai hoặc gửi cho người khác.
